In [2]:
import pandas as pd
import logging
import sys
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

In [4]:
# === НАСТРОЙКА ПУТЕЙ ===
current_dir = Path.cwd()
project_src = current_dir.parent

if str(project_src) not in sys.path:
    sys.path.append(str(project_src))

from app.rag_pipeline import ScienceRAG

# Загружаем переменные окружения
load_dotenv()

True

In [5]:
# === КОНФИГУРАЦИЯ ===
INPUT_FILE = Path('../../data/eval/gd.json')
OUTPUT_FILE = Path('../../data/eval/gd_filled.parquet')
# Сколько чанков будет возвращать ретривер?
TOP_K = 10

# Подавляем INFO логи от httpx
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# === ЗАГРУЗКА ДАННЫХ ===
df = pd.read_json(INPUT_FILE)

In [ ]:
# === ИНИЦИАЛИЗАЦИЯ RAG ===
rag = ScienceRAG()

In [ ]:
# === ФУНКЦИЯ ЗАПОЛНЕНИЯ ===
def get_rag_data(question, top_k):
    """Получает ответ и контекст от RAG системы"""
    res = rag.answer(question, top_k)
    context_list = [source['text'] for source in res.get('sources', [])]
    return pd.Series([res['answer'], context_list])

In [ ]:
# === ОБРАБОТКА ДАТАСЕТА ===
tqdm.pandas()
df[['answer', 'retrieved_contexts']] = df['question'].progress_apply(get_rag_data, top_k=TOP_K)

In [ ]:
# === СОХРАНЕНИЕ РЕЗУЛЬТАТА ===
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_FILE, index=False)

print(f"\nДатасет сохранён: {OUTPUT_FILE}")

In [ ]:
result = pd.read_parquet(OUTPUT_FILE)

In [ ]:
result

In [ ]:
result.iloc[46]['question']

In [ ]:
result.iloc[46]['answer']